# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maahilkureshi-design/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

Before testing anything I look at the shape of the key fields. Search data has heavy tails: a few pages hold most of the impressions, and a median page is much smaller than the average one. The table below shows quantiles, and the figure (log scale) saves to `work/outputs/` for reuse.

In [1]:
import os, json, shutil
import numpy as np
import pandas as pd
import duckdb
from sklearn.metrics import average_precision_score

# The saved snapshot files and shared outputs live in Google Drive (MyDrive/flyrank_capstone),
# so this notebook can run in its own session after capstone.ipynb has finished.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT, ON_COLAB = '/content/drive/MyDrive/flyrank_capstone', True
except ImportError:
    DRIVE_ROOT, ON_COLAB = '.', False
CACHE_DIR, EVAL_DIR, OUT_DIR = 'snap_cache', 'eval_cache', f'{DRIVE_ROOT}/outputs'
if ON_COLAB:
    assert os.path.exists(f'{DRIVE_ROOT}/snap_cache'), 'Not found in Google Drive: run capstone.ipynb first, then its last cell (copy to Drive).'
    shutil.copytree(f'{DRIVE_ROOT}/snap_cache', CACHE_DIR, dirs_exist_ok=True)      # local copy: faster than reading from Drive
    if os.path.exists(f'{DRIVE_ROOT}/eval_cache'):
        shutil.copytree(f'{DRIVE_ROOT}/eval_cache', EVAL_DIR, dirs_exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
n_files = len([f for f in os.listdir(CACHE_DIR) if f.startswith('snap_') and f.endswith('.parquet')]) if os.path.exists(CACHE_DIR) else 0
assert n_files == 12, f'{n_files} snapshot files found, expected 12: run capstone.ipynb and its last cell (copy to Drive) first.'

LABEL_DAYS = 30
K_LIST = [20, 50]
MIN_CLIENT_PAGES_MODEL = 200
SPIKE_RULE = "imp_last30 >= 2 * GREATEST(imp_prev30, imp_prev60)"
UNIVERSE = f"n_elig >= {MIN_CLIENT_PAGES_MODEL} AND NOT ({SPIKE_RULE})"
PAST_COLS = ['imp_90', 'clicks_90', 'ctr_90', 'ctr_last30', 'pos_90', 'pos_last30', 'pos_prev30', 'pos_change',
             'imp_last30', 'imp_prev30', 'imp_prev60', 'active_days_90', 'cv_90',
             'log_mom', 'log_mom_rel', 'log_mom_prev', 'log_mom_prev_rel', 'page_age_days']
STATIC_COLS = ['word_count', 'char_count', 'content_type', 'main_intent', 'competition_level', 'competition',
               'search_volume', 'cpc', 'backlinks', 'category_count', 'keyword_token_count', 'url_char_count']
mcon = duckdb.connect()

def _clean(d):
    d['snap_t'] = pd.to_datetime(d['snap_t'])
    for c in PAST_COLS:
        if pd.api.types.is_timedelta64_dtype(d[c]):
            d[c] = d[c].dt.days
    return d

def load_train():
    cols = ', '.join(['snap_t', 'client_hash_id', 'y'] + PAST_COLS)
    return _clean(mcon.sql(f"SELECT {cols} FROM read_parquet('{CACHE_DIR}/snap_*.parquet') WHERE {UNIVERSE}").df())

def load_eval(name):
    path = f'{EVAL_DIR}/eval_{name}.parquet'
    assert os.path.exists(path), f'{path} is missing: run the build cells of capstone.ipynb first'
    cols = ', '.join(['snap_t', 'client_hash_id', 'y'] + PAST_COLS)
    return _clean(mcon.sql(f"SELECT {cols} FROM read_parquet('{path}') WHERE {UNIVERSE}").df())

def eval_rule(df, score_col, ks=K_LIST):
    rows = []
    for t, g in df.groupby('snap_t'):
        g = g.sort_values(score_col, ascending=False, kind='mergesort')
        base = g['y'].mean()
        row = {'snap_t': str(t)[:10], 'n': len(g), 'base_rate': round(base, 3)}
        for k in ks:
            p = g['y'].head(k).mean()
            row[f'p@{k}'] = round(p, 3)
            row[f'lift@{k}'] = round(p / base, 2) if base > 0 else np.nan
        ap = average_precision_score(g['y'], g[score_col]) if g['y'].sum() > 0 else np.nan
        row['ap'] = round(ap, 3)
        row['ap_lift'] = round(ap / base, 2) if base > 0 else np.nan
        rows.append(row)
    return pd.DataFrame(rows)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

df = load_train()
print(f"{len(df):,} labeled rows, base rate of the label {df['y'].mean():.3f}")

KEY = ['imp_last30', 'imp_90', 'cv_90', 'pos_last30', 'active_days_90', 'page_age_days']
q = df[KEY].quantile([0.01, 0.25, 0.5, 0.75, 0.9, 0.99]).T
q.columns = ['p1', 'p25', 'median', 'p75', 'p90', 'p99']
q['max'] = df[KEY].max()
q['mean_over_median'] = (df[KEY].mean() / df[KEY].median()).round(2)
print(q.round(2).to_string())

fig, axes = plt.subplots(2, 3, figsize=(11, 6))
for ax, c in zip(axes.ravel(), KEY):
    s = df[c].replace([np.inf, -np.inf], np.nan).dropna()
    s = s[s > 0] if c in ('imp_last30', 'imp_90') else s
    ax.hist(np.log10(s + 1) if c in ('imp_last30', 'imp_90') else s, bins=50)
    ax.set_title(c + (' (log10)' if c in ('imp_last30', 'imp_90') else ''))
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'fig_distributions.png'), dpi=120)
plt.show()

Mounted at /content/drive
219,258 labeled rows, base rate of the label 0.102
                    p1     p25   median      p75       p90       p99        max  mean_over_median
imp_last30      104.00  286.00   762.00  2318.75   5780.00  25864.00  239235.00              3.24
imp_90          235.00  857.00  2223.00  6504.00  16687.30  70047.89  509684.00              3.14
cv_90             0.21    0.42     0.57     0.78      1.07      2.06       9.24              1.14
pos_last30        0.93    4.62     7.73    16.59     34.69     72.43      99.76              1.80
active_days_90   49.00   88.00    90.00    90.00     90.00     90.00      90.00              0.96
page_age_days    94.00  145.00   200.00   266.00    353.00    444.00     494.00              1.08


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

Three safe signals, each stated as a hypothesis before the test, then checked on the training snapshots (new or spiking pages excluded, as in the capstone). A page's bucket is its decile of the signal within its snapshot, so months with different base rates are not mixed. The label rate is the share of pages whose impressions fell to half or less of their client's trend over the next 30 days.

| Signal | Hypothesis |
|---|---|
| Volatility (`cv_90`) | More volatile pages are more likely to decline next month |
| Position (`pos_last30`) | Pages at deeper positions (larger numbers) are more likely to decline |
| Active days (`active_days_90`) | Pages visible on fewer days are more likely to decline |

Verdict rule, fixed in advance, comparing the lowest and highest decile with the overall rate: **MIXED** if both ends are at least 1.25x the overall rate; **CONFIRMED** if the hypothesised end is at least 1.25x; **OPPOSITE** if the other end is at least 1.25x; otherwise **FALSE**.

In [2]:
def wilson(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    d = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / d
    half = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (round(centre - half, 3), round(centre + half, 3))

def bucket_table(data, col, q=10):
    d = data[['snap_t', 'y', col]].dropna().copy()
    d['bucket'] = d.groupby('snap_t')[col].transform(lambda s: pd.qcut(s.rank(method='first'), q, labels=False) + 1)
    g = d.groupby('bucket')['y'].agg(['size', 'sum']).reset_index()
    g['label_rate'] = (g['sum'] / g['size']).round(3)
    g['ci95'] = [wilson(int(k), int(n)) for k, n in zip(g['sum'], g['size'])]
    return g[['bucket', 'size', 'label_rate', 'ci95']], d['y'].mean()

def verdict(tab, base, hypothesis):
    lo, hi = tab.iloc[0]['label_rate'], tab.iloc[-1]['label_rate']
    r_lo, r_hi = lo / base, hi / base
    if r_lo >= 1.25 and r_hi >= 1.25:
        return 'MIXED'
    effect, other = (r_hi, r_lo) if hypothesis == 'higher' else (r_lo, r_hi)
    if effect >= 1.25:
        return 'CONFIRMED'
    if other >= 1.25:
        return 'OPPOSITE'
    return 'FALSE'

VERDICTS = []
for col, hyp, text in [('cv_90', 'higher', 'more volatile pages decline more'),
                       ('pos_last30', 'higher', 'deeper positions decline more'),
                       ('active_days_90', 'lower', 'fewer active days decline more')]:
    tab, base = bucket_table(df, col)
    v = verdict(tab, base, hyp)
    VERDICTS.append({'signal': col, 'hypothesis': text, 'verdict': v,
                     'lowest_decile_rate': tab.iloc[0]['label_rate'], 'highest_decile_rate': tab.iloc[-1]['label_rate'],
                     'overall_rate': round(base, 3)})
    print(f"\n{col}: {text} -> {v}")
    print(tab.to_string(index=False))


cv_90: more volatile pages decline more -> CONFIRMED
 bucket  size  label_rate           ci95
      1 21930       0.025 (0.023, 0.027)
      2 21927       0.040 (0.038, 0.043)
      3 21924       0.053  (0.05, 0.056)
      4 21923       0.062 (0.059, 0.065)
      5 21926       0.077  (0.073, 0.08)
      6 21925       0.096   (0.092, 0.1)
      7 21922       0.117 (0.113, 0.122)
      8 21925       0.140 (0.135, 0.144)
      9 21926       0.187 (0.182, 0.192)
     10 21930       0.226 (0.221, 0.232)

pos_last30: deeper positions decline more -> CONFIRMED
 bucket  size  label_rate           ci95
      1 21930       0.092 (0.088, 0.096)
      2 21927       0.065 (0.062, 0.068)
      3 21924       0.072 (0.068, 0.075)
      4 21923       0.080 (0.076, 0.083)
      5 21926       0.097 (0.094, 0.101)
      6 21925       0.100 (0.096, 0.104)
      7 21922       0.106  (0.102, 0.11)
      8 21925       0.106 (0.102, 0.111)
      9 21926       0.120 (0.116, 0.125)
     10 21930       0.184 (0.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

**The flag-linked test.** Many trend flags rest on one assumption: a page that has been falling will keep falling, so a decline is a reason to review it. I test that assumption directly with past relative momentum (`log_mom_rel`: the page's last 30 days against the previous 30, minus its client's trend). If the assumption held, the label rate would rise steadily as momentum gets more negative. In the capstone run the pattern was not a straight line: the label rate was about 18% in the lowest decile (fell most), about 7.5% in the middle, and about 15% in the highest decile (rose most). The test below recomputes it and also checks the narrower claim "two falls in a row".

In [3]:
tab, base = bucket_table(df, 'log_mom_rel')
v = verdict(tab, base, 'lower')
VERDICTS.append({'signal': 'log_mom_rel', 'hypothesis': 'pages that fell most (relative to client) keep falling', 'verdict': v,
                 'lowest_decile_rate': tab.iloc[0]['label_rate'], 'highest_decile_rate': tab.iloc[-1]['label_rate'],
                 'overall_rate': round(base, 3)})
print(f"Flag assumption 'falling pages keep falling' -> {v}")
print(tab.to_string(index=False))

two_falls = (df['log_mom_prev_rel'] <= -0.4) & (df['log_mom_rel'] <= -0.4)
n_two, k_two = int(two_falls.sum()), int(df.loc[two_falls, 'y'].sum())
print(f"\nTwo relative falls in a row: {n_two:,} pages, label rate {k_two / max(n_two, 1):.3f} "
      f"(95% interval {wilson(k_two, n_two)}), overall rate {df['y'].mean():.3f}")

Flag assumption 'falling pages keep falling' -> MIXED
 bucket  size  label_rate           ci95
      1 21930       0.177 (0.172, 0.182)
      2 21927       0.123 (0.118, 0.127)
      3 21924       0.095 (0.092, 0.099)
      4 21923       0.084  (0.08, 0.087)
      5 21926       0.076 (0.072, 0.079)
      6 21925       0.075 (0.071, 0.078)
      7 21922       0.076  (0.073, 0.08)
      8 21925       0.082 (0.078, 0.085)
      9 21926       0.088 (0.084, 0.092)
     10 21930       0.148 (0.143, 0.153)

Two relative falls in a row: 8,283 pages, label rate 0.141 (95% interval (np.float64(0.134), np.float64(0.149))), overall rate 0.102


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

- **Past decline alone is a weak reason to act.** Falling pages are somewhat more likely to fall further, but the effect is concentrated in the most extreme cases, and rising pages also give back ground. A content team should not treat "was falling last month" as a sufficient flag.
- **How a page behaves can matter more than which way it moved.** In the model audit (`w05_model`), volatility and the number of active days were the features the model leaned on most; the verdict table above shows how each signal relates to the label on its own. Volatile, intermittent pages are often low-volume pages where counts are noisy, so a review should check the page's recent history before acting.
- **These are observed associations over 12 monthly snapshots,** with new or spiking pages left out. They support deciding where to look first. They do not show why a page moved, or that any change would help.

In [4]:
print(pd.DataFrame(VERDICTS).to_string(index=False))
pd.DataFrame(VERDICTS).to_csv(os.path.join(OUT_DIR, 'signal_verdicts.csv'), index=False)

        signal                                             hypothesis   verdict  lowest_decile_rate  highest_decile_rate  overall_rate
         cv_90                       more volatile pages decline more CONFIRMED               0.025                0.226         0.102
    pos_last30                          deeper positions decline more CONFIRMED               0.092                0.184         0.102
active_days_90                         fewer active days decline more CONFIRMED               0.246                0.068         0.102
   log_mom_rel pages that fell most (relative to client) keep falling     MIXED               0.177                0.148         0.102


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.